# **Imbalanced Data Handling**

Prerequisites: Probability primer (Bayes' theorem worked example — the
disease-test example in `02-Probability-Statistics-Primer.md` §2 is the
conceptual seed for this whole topic), Model Evaluation (precision/recall).
Directly relevant to `datasets/cardio_modify.csv` and `datasets/kyphosis.csv`
in this repo — both realistically imbalanced medical datasets.

## 1. Why accuracy is misleading - worked numerical example

Dataset: 950 negative, 50 positive (5% positive rate — realistic for many
medical screening scenarios). A trivial "always predict negative"
classifier achieves:
$$\text{Accuracy} = \frac{950}{1000} = 0.95$$
95% accuracy while **never** identifying a single positive case
(Recall $=0/50=0$). This is the precise numeric demonstration of why
accuracy alone is insufficient for imbalanced problems, motivating
precision/recall/F1/PR-AUC (already derived in
`16-Model-Evaluation-Additions.md`) as the metrics that actually matter here.

## 2. Class weighting - derivation

Modify the loss function to penalize minority-class errors more:
$$J_{weighted}(\theta) = \sum_i w_{y_i}\, L(y_i,\hat y_i), \qquad w_k = \frac{n}{K\cdot n_k}$$
(inverse-frequency weighting: rarer classes get proportionally larger
weight). For the 950/50 example: $w_{neg}=\frac{1000}{2(950)}\approx0.526$,
$w_{pos}=\frac{1000}{2(50)}=10.0$ — misclassifying a positive costs 19×
what misclassifying a negative costs, roughly matching the class
imbalance ratio.

## 3. SMOTE (Synthetic Minority Oversampling) - derivation of the interpolation

For each minority-class point $x_i$, pick one of its $k$ nearest
minority-class neighbors $x_{zi}$, generate a synthetic point:
$$x_{new} = x_i + \lambda\,(x_{zi}-x_i), \qquad \lambda\sim\text{Uniform}(0,1)$$
i.e. a random point on the line segment between $x_i$ and a nearby same-class
point — expands the minority class's region in feature space rather than
just duplicating existing points (which plain oversampling does, and which
tends to cause overfitting to those exact duplicated points).

### Worked numerical example

Minority points $x_i=(2,3)$, its nearest minority neighbor $x_{zi}=(4,5)$.
Draw $\lambda=0.3$: $x_{new} = (2,3)+0.3[(4,5)-(2,3)] = (2,3)+0.3(2,2)=(2.6,3.6)$
— a new synthetic minority point strictly between the two originals.

## 4. Threshold moving - derivation

A classifier's default decision threshold (predict positive if
$\hat p>0.5$) is not necessarily optimal for imbalanced costs. Choose the
threshold $t^*$ that maximizes a cost-weighted objective (e.g. F1, or an
explicit cost matrix) by sweeping $t$ over the PR curve
(`16-Model-Evaluation-Additions.md`'s ROC/PR derivation) rather than
defaulting to 0.5 — cheap, requires no retraining, often the single
highest-leverage fix.

## 5. Python - verify the accuracy-paradox and SMOTE worked examples

In [1]:
import numpy as np

# Accuracy paradox
y_true = np.array([0]*950 + [1]*50)
y_pred = np.zeros(1000)   # always predict negative
acc = (y_true == y_pred).mean()
recall = ((y_pred==1) & (y_true==1)).sum() / (y_true==1).sum() if (y_true==1).sum() else 0
print(acc, recall)   # 0.95  0.0 (guard against div-by-zero if no positives predicted)

# SMOTE interpolation
xi, xzi, lam = np.array([2,3]), np.array([4,5]), 0.3
x_new = xi + lam*(xzi - xi)
print(x_new)   # [2.6 3.6]

# Class weights
w_neg = 1000/(2*950); w_pos = 1000/(2*50)
print(w_neg, w_pos, w_pos/w_neg)   # 0.526  10.0  ratio ~19

0.95 0.0
[2.6 3.6]
0.5263157894736842 10.0 19.0
